# Orthogonalization Experiemnts
This colab shows the code setup we used for the orthogonalization experiments discussed in section 4.2 of our paper.

#Setup
Code for imports and model setup

In [ ]:
import torch, itertools, numpy as np, pandas as pd
import torch.nn as nn, torch.optim as optim
from scipy.stats import pearsonr
import pandas as pd
import seaborn as sns, matplotlib.pyplot as plt



class TinyMLP(nn.Module):
    """
    Simple MLP for XOR:
    2 inputs → hidden layer (ReLU) → 1 output (sigmoid)
    """
    def __init__(self, in_dim=2, hidden_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        return torch.sigmoid(self.fc2(torch.relu(self.fc1(x))))

def make_xor_data():
    """
    Generates XOR dataset:
    """
    X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], requires_grad=True)
    y = torch.tensor([[0.],[1.],[1.],[0.]], requires_grad=True)
    return X, y

def train_model_orth(model, X, y, l1=0.0, lr=0.05, epochs=5000, lambda_orth=0.0):
    """
    Trains model using Adam optimizer and binary cross-entropy loss.
    Applies optional l1 regularization and orthogonalization regularization
    """
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    for _ in range(epochs):
        opt.zero_grad()
        y_pred = model(X)
        bce = loss_fn(y_pred, y)
        l1_term = model.fc1.weight.abs().sum() * l1
        W = model.fc1.weight
        orth_loss = ((W @ W.T - torch.eye(W.shape[0]))**2).sum() * lambda_orth
        loss = bce + l1_term + orth_loss
        loss.backward()
        opt.step()
    return model

# Experiment setup

Code for computing effective rank, condition number, and multiplicity of circuit

In [ ]:
def functional_similarity(model, circuit, X, ref_outputs):
    """
    Measures how well a given neuron subset ("circuit") reproduces
    the model's original outputs.

    Process:
    - Creates a mask based off of circuit to keep only select neurons active
    - Uses a hook to zero out anything not part of the circuit
    - Computes model output for only the circuit
    - Calculates Normalized Similarity and Pearson Correlation with reference output

    Args:
      model: MLP model
      circuit: Iterable of neuron indices representing circuit
      X: Input data (XOR dataset)
      ref_outputs: Original model outputs used as reference

    Returns:
      norm_sim : Normalized similarity between circuit output
              and reference output
      corr : Pearson correlation between circuit output and
            reference output
    """
    with torch.no_grad():
        mask = torch.zeros(model.fc1.out_features)
        mask[list(circuit)] = 1.0
        def hook(module, inp, out): return out * mask
        h = model.fc1.register_forward_hook(hook)
        out = model(X)
        h.remove()
        mse = torch.mean((out - ref_outputs)**2).item()
        var_ref = torch.var(ref_outputs).item() + 1e-8
        norm_sim = 1 - mse / var_ref
        corr, _ = pearsonr(out.flatten().numpy(), ref_outputs.flatten().numpy())
    return max(0, min(1, norm_sim)), corr

def circuit_overlap(circuits):
    """
    Computes mean overlap for circuit

    Args:
      circuits: List of circuits

    Returns:
      mean overlap: Average overlap across all circuit pairs
    """
    overlaps = []
    for i in range(len(circuits)):
        for j in range(i+1, len(circuits)):
            A,B = set(circuits[i]), set(circuits[j])
            overlaps.append(len(A & B)/len(A | B))
    return np.mean(overlaps) if overlaps else 0.0

def enumerate_circuits(model, X):
    """
    Enumerates through every possible circuit in model and evaluates output of circuit on dataset X.
    Calculates functional similarity and correlation of each circuit

    Args:
      model: MLP model
      X: Input data (XOR dataset)

    Returns:
      results: pd dataframe table containing each circuit along with its
            functional similarity and correlation to the full model output
    """

    with torch.no_grad():
        ref_outputs = model(X)
    hidden_dim = model.fc1.out_features
    all_circuits = []
    for k in range(1, hidden_dim+1):
        for subset in itertools.combinations(range(hidden_dim), k):
            all_circuits.append(subset)
    results = []
    for c in all_circuits:
        sim, corr = functional_similarity(model, c, X, ref_outputs)
        results.append({"circuit": c, "similarity": sim, "corr": corr})
    return pd.DataFrame(results)

def activation_condition(model, X):
    """
    Analyze condition number of activation matrix

    Args:
        model: MLP model
        X: Input data (XOR dataset)

    Returns:
        cond_num : Condition number of activation matrix
    """

    with torch.no_grad():
        H = torch.relu(model.fc1(X)).numpy()
    _, s, _ = np.linalg.svd(H)
    cond_num = s.max() / (s.min() + 1e-8)
    return cond_num

In [ ]:
def run_orthogonalization_experiment(X, y, lambdas=[0, 1e-3, 1e-2, 1e-1], widths=[3,4,5,6], l1=0.0):
    """
    Runs experiment and gathers data on different MLPs with varying widths, seeds, and orthogonalization regularization strengths

    Args:

    Returns:
    - Hidden layer width
    - Orthogonalization
    - Number of valid circuits
    - Condition number
    (In form of pd dataframe)

    """
    records = []
    for lam in lambdas:
        for hd in widths:
            torch.manual_seed(0)
            model = TinyMLP(2, hd)
            model = train_model_orth(model, X, y, l1=l1, lambda_orth=lam)
            cond = activation_condition(model, X)
            df_temp = enumerate_circuits(model, X)
            valid = df_temp.query("similarity >= 0.9")
            records.append({
                "hidden_dim": hd,
                "lambda_orth": lam,
                "num_valid": len(valid),
                "cond": cond,
            })
    return pd.DataFrame(records)


#Running Experiment on Orthogonalization and Displaying Data

In [ ]:
X, y = make_xor_data()

df_orth = run_orthogonalization_experiment(X, y)

import seaborn as sns, matplotlib.pyplot as plt
sns.lineplot(data=df_orth, x="lambda_orth", y="num_valid", hue="hidden_dim")
plt.title("Orthogonalization vs Mechanistic Multiplicity"); plt.show()
